# Client B: DEAD (Drinking Excess Alcohol is Dangerous) — Zip-Month Purchase Drivers

**Client:** DEAD, an Iowa nonprofit working on safer alcohol culture. **Their question:**
what factors drive higher or lower alcohol purchases, by area and by month.

**Target: liters per capita**, from zip population pulled via the Census API in section 5
-- not raw liters (confounded by zip size) and not dollars (confounded by price).

**Read section 1b first: this data is almost certainly spirits only.** Iowa's Alcoholic
Beverages Division has held exclusive wholesale control of *spirits* since 1934, but gave
up control of wine in 1985 and never controlled beer -- both are distributed privately and
don't appear to be reflected in `category_name` here. If that holds, **none of this
notebook says anything about beer or wine purchases**, which matters a lot for a client
concerned with alcohol broadly, not spirits specifically. Section 1b checks this against
the actual category list rather than asserting it.

**Two views, not one:** section 11 gives statewide "when" drivers; section 12 gives a
"where" companion for Iowa City / Ames specifically, since a pooled statewide model
structurally cannot see effects that are large in two college towns and near-zero
everywhere else (the Hawkeyes/Cyclones/Iowa State Fair coefficients in section 11 are a
symptom of this, not evidence the effects don't exist).

In [ ]:
import os
import getpass
import duckdb as db
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

con = db.connect()
RAW = 'liquor_2022_2026.parquet'
CENSUS = 'liquor_census.parquet'

In [ ]:
con = db.connect()

RAW = 'liquor_2022_2026.parquet'
CENSUS = 'liquor_census.parquet'

## 1. Clean zip codes and build the zip x month sales aggregate

In [ ]:
zip_month = con.execute(f'''
    WITH cleaned AS (
        SELECT
            SUBSTR(TRIM(store_zip_code), 1, 5) AS zip,
            ordered_on,
            strftime(ordered_on, '%Y-%m') AS year_month,
            TRY_CAST(sales_bottles AS DOUBLE) AS sales_bottles,
            sales_dollars,
            sales_liters,
            store_no,
            category_name,
            TRY_CAST(state_bottle_retail AS DOUBLE) AS state_bottle_retail,
            bottle_volume_ml
        FROM '{RAW}'
        WHERE SUBSTR(TRIM(store_zip_code), 1, 5) SIMILAR TO '[0-9]{{5}}'
    )
    SELECT
        zip, year_month,
        SUM(sales_dollars) AS total_dollars,
        SUM(sales_bottles) AS total_bottles,
        SUM(sales_liters) AS total_liters,
        COUNT(DISTINCT store_no) AS n_stores,
        COUNT(DISTINCT category_name) AS n_categories,
        COUNT(*) AS n_line_items,
        AVG(state_bottle_retail) AS avg_bottle_retail,
        AVG(bottle_volume_ml) AS avg_bottle_volume_ml
    FROM cleaned
    GROUP BY zip, year_month
''').df()

zip_month["year"] = zip_month["year_month"].str[:4].astype(int)
zip_month["month_num"] = zip_month["year_month"].str[5:7].astype(int)

n_neg_liters = (zip_month["total_liters"] < 0).sum()
n_neg_dollars = (zip_month["total_dollars"] < 0).sum()
print(f"{n_neg_liters} zip-month rows have negative net liters, {n_neg_dollars} negative net dollars "
      f"(returns > sales that month); both clipped to 0.")
zip_month["total_liters"] = zip_month["total_liters"].clip(lower=0)
zip_month["total_dollars"] = zip_month["total_dollars"].clip(lower=0)

print(f"{len(zip_month):,} zip-month rows, {zip_month['zip'].nunique()} distinct zips, "
      f"{zip_month['year_month'].nunique()} months")
zip_month.head()

## 1b. Is this actually spirits-only? Check, don't assume.

Iowa privatized wine wholesale in 1985 and never controlled beer at all -- only spirits
have stayed under state wholesale control. If `liquor_2022_2026.parquet` comes from that
system, `category_name` should contain zero beer or wine entries. Checked directly below.

In [ ]:
categories_seen = con.execute(f"SELECT DISTINCT category_name FROM '{RAW}' ORDER BY category_name").df()
print(f"{len(categories_seen)} distinct category_name values in the raw data:")
for c in categories_seen["category_name"]:
    print(" ", c)

beer_or_wine = categories_seen["category_name"].str.contains("BEER|WINE|MALT", case=False, na=False)
matches = categories_seen.loc[beer_or_wine, "category_name"].tolist()
if matches:
    print(f"\nFOUND beer/wine/malt-related categories -- the spirits-only assumption above is WRONG: {matches}")
else:
    print("\nNo beer/wine/malt categories found -- confirms this dataset is spirits only. "
          "Everything downstream (including the Super Bowl Sunday coefficient in section 11) "
          "should be read as 'spirits purchases,' not 'alcohol purchases' generally.")

## 2. Category-mix features per zip-month

Dollar share of the top categories (statewide ranking, from `liquor_census.parquet`) plus
an "other" bucket. A **predictor** describing purchase composition, not the outcome.

In [ ]:
top_categories = con.execute(f'''
    SELECT category_name, SUM(total_dollars) AS statewide_dollars
    FROM '{CENSUS}'
    WHERE category_name != ''
    GROUP BY category_name
    ORDER BY statewide_dollars DESC
    LIMIT 10
''').df()["category_name"].tolist()

cat_long = con.execute(f'''
    SELECT
        SUBSTR(TRIM(store_zip_code), 1, 5) AS zip,
        strftime(ordered_on, '%Y-%m') AS year_month,
        CASE WHEN category_name IN ({','.join(f"'{c}'" for c in top_categories)})
             THEN category_name ELSE 'OTHER' END AS category_bucket,
        SUM(sales_dollars) AS category_dollars
    FROM '{RAW}'
    WHERE SUBSTR(TRIM(store_zip_code), 1, 5) SIMILAR TO '[0-9]{{5}}'
    GROUP BY zip, year_month, category_bucket
''').df()

cat_wide = cat_long.pivot_table(
    index=["zip", "year_month"], columns="category_bucket", values="category_dollars",
    aggfunc="sum", fill_value=0,
).reset_index()
cat_wide.columns = ["zip", "year_month"] + [f"share_{c.lower().replace(' ', '_')}" for c in cat_wide.columns[2:]]
share_cols = [c for c in cat_wide.columns if c.startswith("share_")]

zip_month = zip_month.merge(cat_wide, on=["zip", "year_month"], how="left")
zip_month[share_cols] = zip_month[share_cols].div(zip_month["total_dollars"].replace(0, np.nan), axis=0).fillna(0)

print("category buckets:", share_cols)
zip_month[["zip", "year_month"] + share_cols].head()

--gabby stuff cuz idk why above wasnt running^^

In [ ]:
top_categories = con.execute(f'''
    SELECT 
        category_name, 
        SUM(sales_dollars) AS statewide_dollars
    FROM '{CENSUS}'
    WHERE category_name IS NOT NULL
      AND category_name != ''
    GROUP BY category_name
    ORDER BY statewide_dollars DESC
    LIMIT 10
''').df()["category_name"].tolist()

cat_long = con.execute(f'''
    SELECT
        SUBSTR(TRIM(store_zip_code), 1, 5) AS zip,
        strftime(ordered_on, '%Y-%m') AS year_month,

        CASE 
            WHEN category_name IN ({','.join(f"'{c}'" for c in top_categories)})
            THEN category_name 
            ELSE 'OTHER' 
        END AS category_bucket,

        SUM(sales_dollars) AS category_dollars

    FROM '{CENSUS}'

    WHERE SUBSTR(TRIM(store_zip_code), 1, 5) SIMILAR TO '[0-9]{{5}}'

    GROUP BY zip, year_month, category_bucket
''').df()

cat_wide = cat_long.pivot_table(
    index=["zip", "year_month"],
    columns="category_bucket",
    values="category_dollars",
    aggfunc="sum",
    fill_value=0
).reset_index()

cat_wide.columns = (
    ["zip", "year_month"] +
    [f"share_{c.lower().replace(' ', '_')}" for c in cat_wide.columns[2:]]
)

share_cols = [
    c for c in cat_wide.columns
    if c.startswith("share_")
]

zip_month = zip_month.merge(
    cat_wide,
    on=["zip", "year_month"],
    how="left"
)

zip_month[share_cols] = (
    zip_month[share_cols]
    .div(
        zip_month["total_dollars"].replace(0, np.nan),
        axis=0
    )
    .fillna(0)
)

print("category buckets:", share_cols)

zip_month[
    ["zip", "year_month"] + share_cols
].head()

## 3. Price-premium feature (vs. statewide, from `liquor_census.parquet`)

In [ ]:
statewide_price = con.execute(f"""
    SELECT
        strftime(ordered_on, '%Y-%m') AS year_month,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE)) AS statewide_avg_bottle_price
    FROM '{CENSUS}'
    WHERE ordered_on IS NOT NULL
    GROUP BY strftime(ordered_on, '%Y-%m')
    ORDER BY year_month
""").df()

zip_month = zip_month.merge(
    statewide_price,
    on="year_month",
    how="left"
)

zip_month["price_premium_vs_state"] = (
    zip_month["avg_bottle_retail"]
    / zip_month["statewide_avg_bottle_price"]
    - 1
)

statewide_price.head()

In [ ]:
# Remove old statewide price columns if this cell was run before
old_cols = [
    "statewide_avg_bottle_price",
    "statewide_avg_bottle_price_x",
    "statewide_avg_bottle_price_y",
    "price_premium_vs_state"
]

zip_month = zip_month.drop(
    columns=[c for c in old_cols if c in zip_month.columns]
)


# Calculate statewide average bottle price by month
statewide_price = con.execute(f"""
    SELECT
        strftime(ordered_on, '%Y-%m') AS year_month,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE)) 
            AS statewide_avg_bottle_price
    FROM '{CENSUS}'
    WHERE ordered_on IS NOT NULL
      AND TRY_CAST(state_bottle_retail AS DOUBLE) IS NOT NULL
    GROUP BY strftime(ordered_on, '%Y-%m')
    ORDER BY year_month
""").df()


# Make sure year_month has the same type in both dataframes
zip_month["year_month"] = zip_month["year_month"].astype(str)
statewide_price["year_month"] = statewide_price["year_month"].astype(str)


# Merge statewide monthly price into ZIP-month data
zip_month = zip_month.merge(
    statewide_price,
    on="year_month",
    how="left"
)


# Create engineered price feature
zip_month["price_premium_vs_state"] = (
    zip_month["avg_bottle_retail"]
    / zip_month["statewide_avg_bottle_price"]
    - 1
)


# Check result
zip_month[
    [
        "store_zip_code",
        "year_month",
        "avg_bottle_retail",
        "statewide_avg_bottle_price",
        "price_premium_vs_state"
    ]
].head(10)

In [ ]:
statewide_price = con.execute(f'''
    SELECT
        strftime(ordered_on, '%Y-%m') AS year_month,
        AVG(state_bottle_retail) AS statewide_avg_bottle_price
    FROM '{CENSUS}'
    WHERE category_name IS NOT NULL
      AND category_name != ''
    GROUP BY year_month
    ORDER BY year_month
''').df()

zip_month = zip_month.merge(
    statewide_price,
    on="year_month",
    how="left"
)

zip_month["price_premium_vs_state"] = (
    zip_month["avg_bottle_retail"] /
    zip_month["statewide_avg_bottle_price"] - 1
)

zip_month["price_premium_vs_state"] = (
    zip_month["price_premium_vs_state"].fillna(0)
)

zip_month[
    [
        "zip",
        "year_month",
        "avg_bottle_retail",
        "statewide_avg_bottle_price",
        "price_premium_vs_state"
    ]
].head()

In [ ]:
con.execute(f"DESCRIBE '{CENSUS}'").df()

## 4. Calendar events (same importer as `caseb_calendar_events.ipynb`, month grain)

Day-count features rather than one dummy per event: every fixed event maps to exactly one
calendar month every year, so an event dummy and a month-of-year dummy carry the same
information at this grain (see `liquor_census_build_data.ipynb`, section 4).

In [ ]:
def nth_weekday(year, month, weekday, n):
    first = pd.Timestamp(year=year, month=month, day=1)
    return first + pd.Timedelta(days=(weekday - first.weekday()) % 7 + 7 * (n - 1))

def last_weekday(year, month, weekday):
    month_end = pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    return month_end - pd.Timedelta(days=(month_end.weekday() - weekday) % 7)

events = {}
def add_event(date, name):
    events.setdefault(pd.Timestamp(date), []).append(name)

SUPER_BOWL_SUNDAY = {2022: "2022-02-13", 2023: "2023-02-12", 2024: "2024-02-11",
                      2025: "2025-02-09", 2026: "2026-02-08"}
IOWA_STATE_FAIR = {2022: ("2022-08-11", "2022-08-21"), 2023: ("2023-08-10", "2023-08-20"),
                    2024: ("2024-08-08", "2024-08-18"), 2025: ("2025-08-07", "2025-08-17"),
                    2026: ("2026-08-13", "2026-08-23")}
HAWKEYES_HOME = {
    2022: ["2022-09-03", "2022-09-10", "2022-09-17", "2022-10-01", "2022-10-29", "2022-11-12", "2022-11-25"],
    2023: ["2023-09-02", "2023-09-16", "2023-09-30", "2023-10-07", "2023-10-21", "2023-11-11", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-07", "2024-09-14", "2024-10-12", "2024-10-26", "2024-11-02", "2024-11-29"],
    2025: ["2025-08-30", "2025-09-13", "2025-09-27", "2025-10-18", "2025-10-25", "2025-11-08", "2025-11-22"],
    2026: ["2026-09-05"],
}
CYCLONES_HOME = {
    2022: ["2022-09-03", "2022-09-17", "2022-09-24", "2022-10-08", "2022-10-29", "2022-11-05", "2022-11-19"],
    2023: ["2023-09-02", "2023-09-09", "2023-09-23", "2023-10-07", "2023-11-04", "2023-11-18"],
    2024: ["2024-08-31", "2024-09-21", "2024-10-05", "2024-10-19", "2024-11-02", "2024-11-09", "2024-11-16"],
    2025: ["2025-08-30", "2025-09-06", "2025-09-27", "2025-10-25", "2025-11-01", "2025-11-22"],
    2026: ["2026-09-05", "2026-09-12"],
}

for year in range(2022, 2027):
    add_event(f"{year}-01-01", "New Year's Day")
    add_event(f"{year}-12-31", "New Year's Eve")
    add_event(SUPER_BOWL_SUNDAY[year], "Super Bowl Sunday")
    add_event(f"{year}-03-17", "St. Patrick's Day")
    add_event(f"{year}-05-05", "Cinco de Mayo")
    add_event(last_weekday(year, 5, 0), "Memorial Day")
    add_event(f"{year}-07-04", "July 4th")
    add_event(nth_weekday(year, 9, 0, 1), "Labor Day")
    add_event(f"{year}-10-31", "Halloween")
    thanksgiving = nth_weekday(year, 11, 3, 4)
    add_event(thanksgiving, "Thanksgiving")
    add_event(thanksgiving - pd.Timedelta(days=1), "Thanksgiving")
    add_event(f"{year}-12-24", "Christmas")
    add_event(f"{year}-12-25", "Christmas")
    fair_start, fair_end = IOWA_STATE_FAIR[year]
    for day in pd.date_range(fair_start, fair_end):
        add_event(day, "Iowa State Fair")
for date in [d for season in HAWKEYES_HOME.values() for d in season]:
    add_event(date, "Hawkeyes Home Game")
for date in [d for season in CYCLONES_HOME.values() for d in season]:
    add_event(date, "Cyclones Home Game")

events_df = pd.DataFrame([(d, n) for d, n in sorted(events.items())], columns=["date", "events"])
events_df["year_month"] = events_df["date"].dt.strftime("%Y-%m")

EVENT_TYPES = ["New Year's Day", "New Year's Eve", "Super Bowl Sunday", "St. Patrick's Day",
               "Cinco de Mayo", "Memorial Day", "July 4th", "Labor Day", "Halloween",
               "Thanksgiving", "Christmas", "Iowa State Fair", "Hawkeyes Home Game", "Cyclones Home Game"]

def slugify(name):
    return name.lower().replace(" ", "_").replace(".", "").replace("'", "")

month_events = pd.DataFrame({"year_month": sorted(zip_month["year_month"].unique())})
for event_type in EVENT_TYPES:
    mask = events_df["events"].apply(lambda lst: event_type in lst)
    day_counts = events_df.loc[mask].groupby("year_month").size()
    month_events[f"{slugify(event_type)}_days"] = month_events["year_month"].map(day_counts).fillna(0).astype(int)

event_day_cols = [c for c in month_events.columns if c.endswith("_days")]
month_events["n_event_types_in_month"] = (month_events[event_day_cols] > 0).sum(axis=1)

zip_month = zip_month.merge(month_events, on="year_month", how="left")
print(zip_month.shape)
zip_month.head()

## 5. Zip population from the Census API (ACS 5-year, ZCTA)

`store_zip_code` is a USPS ZIP; population is published by **ZCTA**, a Census-drawn
approximation of a ZIP's boundary. Most line up closely, but not always (checked below,
not assumed).

**API key:** don't paste it into this notebook or into chat. Set it as an environment
variable before starting Jupyter (`export CENSUS_API_KEY=...`), and the cell below picks
it up automatically; otherwise it prompts with `getpass` so it's never saved in the
notebook's output.

In [ ]:
CENSUS_API_KEY = "15bd77d8febe3db1e8079c8fd196756e89c4f965"
ACS_YEAR = 2024  # 2020-2024 ACS 5-year estimates

resp = requests.get(
    f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5",
    params={"get": "NAME,B01003_001E", "for": "zip code tabulation area:*", "key": CENSUS_API_KEY},
    timeout=60,
)
resp.raise_for_status()
rows = resp.json()
zcta_pop = pd.DataFrame(rows[1:], columns=rows[0]).rename(
    columns={"zip code tabulation area": "zip", "B01003_001E": "population"}
)[["zip", "population"]]
zcta_pop["population"] = pd.to_numeric(zcta_pop["population"], errors="coerce")

print(f"{len(zcta_pop):,} ZCTAs returned nationwide")

zip_month = zip_month.merge(zcta_pop, on="zip", how="left")
zips_in_data = zip_month["zip"].nunique()
zips_missing = zip_month.loc[zip_month["population"].isna(), "zip"].nunique()
print(f"{zips_missing} of {zips_in_data} zips in the sales data have no matching ZCTA population "
      f"-- left as NaN, not imputed. Rows for those zips are dropped before modeling (next cell).")

In [ ]:
# ACS 5-year estimates for very small ZCTAs carry large margins of error, and a per-capita rate
# divided by a tiny population is dominated by noise rather than signal. Floor at 100 residents.
POP_FLOOR = 100
before = len(zip_month)
zip_month = zip_month[zip_month["population"].notna()].copy()
n_dropped_no_pop = before - len(zip_month)
zip_month = zip_month[zip_month["population"] >= POP_FLOOR].copy()
n_dropped_small = before - n_dropped_no_pop - len(zip_month)
print(f"dropped {n_dropped_no_pop:,} rows (no ZCTA population match), "
      f"{n_dropped_small:,} more rows (ZCTA population under {POP_FLOOR})")
print(f"{len(zip_month):,} zip-month rows remain, {zip_month['zip'].nunique()} zips")

zip_month["liters_per_capita"] = zip_month["total_liters"] / zip_month["population"]
zip_month["log_percapita"] = np.log1p(zip_month["liters_per_capita"])
zip_month[["zip", "year_month", "total_liters", "population", "liters_per_capita"]].head()

## 6. Data limitations & ethical considerations -- read before the results below

**Scope of the data (see section 1b):** if no beer/wine categories turned up, everything
below is spirits-only. That's the single biggest interpretive caveat in this notebook --
it likely explains why Super Bowl Sunday shows almost no effect in section 11 despite
being a famously beer-heavy occasion, and it means no claim here should be generalized to
"alcohol purchases" without that qualifier.

**Still missing:**

- **Alcohol content (ABV/proof).** `liters_per_capita` is liquid volume per resident, not
  ethanol per resident.
- **Income and ethnicity by zip.** Still not in any file used here.
- **Sales \u2260 consumption.** Wholesale distribution timing, not drinking-day timing.

**Caveats that come with population:**

- **ZCTA \u2260 ZIP**, and small-area ACS estimates carry real uncertainty even after the
  >=100-resident floor in section 5.

**A structural caveat about the statewide model (see sections 11-12):** pooling ~900 zip
codes into one regression can only detect effects that are broadly similar everywhere. An
effect that's large in two college towns and absent elsewhere gets averaged toward zero,
not correctly estimated as "zero, except locally." Section 12 checks this directly for
Iowa City and Ames rather than trusting the pooled coefficient at face value.

**Ethical and legal considerations for how DEAD uses this:**

- **Ecological fallacy.** A zip-month correlation describes an area-level pattern, not
  individual drinking behavior; purchases in a zip include visitors and commuters.
- **Demographic data, if added later, needs real safeguards** -- aggregated only, paired
  with per-capita context, and used to guide voluntary resources, never to target or
  penalize a community.
- **Recommended use of this notebook's output:** section 11 as a *timing* signal
  (statewide, spirits-specific), section 12 as a *where* signal for two specific named
  areas with an actual, checkable local effect -- not a tool to publicly rank or name other
  "problem" zip codes on the strength of the pooled model alone.

## 7. Train / dev / test split (dev + test from 2026)

Train is every month before 2026; dev is Jan-Apr 2026; test is May-Aug 2026. The zip's own
baseline average (`zip_avg_log_percapita`) is computed **from train only** and merged
onto dev/test to avoid leakage.

In [ ]:
zip_month["t"] = (zip_month["year"] - zip_month["year"].min()) * 12 + zip_month["month_num"]
zip_month["t"] = zip_month["t"] - zip_month["t"].min()

all_months = sorted(zip_month["year_month"].unique())
months_2026 = [m for m in all_months if m.startswith("2026")]
train_months = set(m for m in all_months if not m.startswith("2026"))
split_point = len(months_2026) // 2
dev_months = set(months_2026[:split_point])
test_months = set(months_2026[split_point:])

train = zip_month[zip_month["year_month"].isin(train_months)].copy()
dev = zip_month[zip_month["year_month"].isin(dev_months)].copy()
test = zip_month[zip_month["year_month"].isin(test_months)].copy()

zip_baseline = train.groupby("zip")["log_percapita"].mean().rename("zip_avg_log_percapita")
global_baseline = train["log_percapita"].mean()
for d in (train, dev, test):
    d["zip_avg_log_percapita"] = d["zip"].map(zip_baseline).fillna(global_baseline)

feature_cols = (
    ["t", "n_stores", "n_categories", "n_line_items", "avg_bottle_volume_ml",
     "price_premium_vs_state", "zip_avg_log_percapita"]
    + share_cols
    + event_day_cols
    + ["n_event_types_in_month"]
)

X_train, y_train = train[feature_cols].fillna(0), train["log_percapita"]
X_dev, y_dev = dev[feature_cols].fillna(0), dev["log_percapita"]
X_test, y_test = test[feature_cols].fillna(0), test["log_percapita"]

print(f"train: {len(train):,} rows ({min(train_months)}..{max(train_months)})")
print(f"dev:   {len(dev):,} rows, 2026 months {sorted(dev_months)}")
print(f"test:  {len(test):,} rows, 2026 months {sorted(test_months)}")
print(f"{len(feature_cols)} features:", feature_cols)

## 8. Train models

A zip-average baseline plus three linear models, all predicting
`log1p(liters_per_capita)`, used for absolute-level accuracy (sections 8-10). Section 11
refits a separate model for driver interpretation -- see that section for why.

In [ ]:
def evaluate(model, X, y, label):
    pred = model.predict(X)
    return {
        "model": label,
        "RMSE_log": mean_squared_error(y, pred) ** 0.5,
        "MAE_log": mean_absolute_error(y, pred),
        "R2_log": r2_score(y, pred),
        "RMSE_liters_percapita": mean_squared_error(np.expm1(y), np.expm1(pred)) ** 0.5,
        "MAE_liters_percapita": mean_absolute_error(np.expm1(y), np.expm1(pred)),
    }

class ZipBaseline:
    def fit(self, X, y): return self
    def predict(self, X): return X["zip_avg_log_percapita"].values

models = {
    "Zip-average baseline": ZipBaseline().fit(X_train, y_train),
    "LinearRegression": LinearRegression().fit(X_train, y_train),
    "Ridge (alpha=1.0)": Ridge(alpha=1.0).fit(X_train, y_train),
    "Lasso (alpha=0.01)": Lasso(alpha=0.01, max_iter=10000).fit(X_train, y_train),
}

dev_results = [evaluate(m, X_dev, y_dev, label) | {"split": "dev"} for label, m in models.items()]
results_df = pd.DataFrame(dev_results).set_index(["model", "split"]).round(4)
results_df

## 9. Test-set evaluation (touched once)

In [ ]:
test_results = [evaluate(m, X_test, y_test, label) | {"split": "test"} for label, m in models.items()]
test_results_df = pd.DataFrame(test_results).set_index(["model", "split"]).round(4)
test_results_df

## 10. Within-zip R\u00b2 -- how much do the features explain beyond "which zip is this"

Subtracting each zip's train-period average from both actual and predicted values
isolates month-to-month movement within a zip. This is a diagnostic on the section-8
models; section 11 fits a dedicated model on this same idea for interpretation.

In [ ]:
def within_zip_r2(df, pred):
    resid_actual = df["log_percapita"] - df["zip_avg_log_percapita"]
    resid_pred = pred - df["zip_avg_log_percapita"]
    return r2_score(resid_actual, resid_pred)

within_rows = [{
    "model": label,
    "within_zip_R2_dev": within_zip_r2(dev, m.predict(X_dev)),
    "within_zip_R2_test": within_zip_r2(test, m.predict(X_test)),
} for label, m in models.items()]
pd.DataFrame(within_rows).set_index("model").round(4)

## 11. Statewide drivers ("when"): a dedicated model, without the zip-baseline bar

The full model in section 8 includes `zip_avg_log_percapita` as a feature -- necessary
for accurate absolute-level prediction, but it dominates a coefficient plot so completely
that every other factor looks negligible next to it, and "some zips just buy more than
others" isn't an actionable finding for a campaign anyway.

**Fix: fit Ridge directly on the *residual* target** (`log_percapita` minus each zip's own
train-period baseline), dropping the baseline from the feature list entirely. This is the
same quantity section 10's within-zip R\u00b2 was already evaluating -- fitting it directly,
rather than deriving it after the fact from a model trained on the full target, gives a
model actually optimized for "what explains this month's deviation from normal," which is
what belongs in a coefficient chart aimed at DEAD.

In [ ]:
driver_feature_cols = [c for c in feature_cols if c != "zip_avg_log_percapita"]

train_resid_y = y_train - X_train["zip_avg_log_percapita"]
dev_resid_y = y_dev - X_dev["zip_avg_log_percapita"]
test_resid_y = y_test - X_test["zip_avg_log_percapita"]

driver_model = Ridge(alpha=1.0).fit(train[driver_feature_cols].fillna(0), train_resid_y)

print("driver model R\u00b2 on within-zip movement:")
print("  dev: ", r2_score(dev_resid_y, driver_model.predict(dev[driver_feature_cols].fillna(0))))
print("  test:", r2_score(test_resid_y, driver_model.predict(test[driver_feature_cols].fillna(0))))

coef_table = pd.DataFrame({
    "coefficient (log scale)": driver_model.coef_,
    "approx. % change in liters/capita per unit": (np.exp(driver_model.coef_) - 1) * 100,
}, index=driver_feature_cols).sort_values("approx. % change in liters/capita per unit", key=abs, ascending=False)
coef_table.round(3)

In [ ]:
plot_data = coef_table.sort_values("coefficient (log scale)")
fig, ax = plt.subplots(figsize=(8, max(4, len(plot_data) * 0.3)))
colors = ["tab:red" if v < 0 else "tab:blue" for v in plot_data["coefficient (log scale)"]]
ax.barh(plot_data.index, plot_data["coefficient (log scale)"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Ridge coefficient (log scale, within-zip movement)")
ax.set_title("Statewide 'when' drivers -- within-zip deviation from normal")
plt.tight_layout()
plt.show()

## 12. Iowa City / Ames ("where"): does pooling wash out a local effect?

Exploratory, not validated the same way the statewide model was -- roughly a dozen zips
across ~56 months is too little data for a held-out split, so this is fit on everything
available and read directionally, not as a tuned predictive model. The zip lists below are
the standard residential/campus ZIPs for each city (approximate by nature of ZIP
boundaries) and are checked against what's actually present in the data, not assumed.

In [ ]:
IOWA_CITY_ZIPS = {"52240", "52241", "52242", "52243", "52245", "52246"}
AMES_ZIPS = {"50010", "50011", "50012", "50013", "50014"}
college_town_zips = IOWA_CITY_ZIPS | AMES_ZIPS

college_town_data = zip_month[zip_month["zip"].isin(college_town_zips)].copy()
present = sorted(college_town_data["zip"].unique())
print(f"{len(present)} of {len(college_town_zips)} target zips present after the population floor: {present}")
print(f"{len(college_town_data):,} zip-month rows")

if len(college_town_data) > 30:
    ct_zip_avg = college_town_data.groupby("zip")["log_percapita"].transform("mean")
    ct_y_resid = college_town_data["log_percapita"] - ct_zip_avg
    ct_X = college_town_data[driver_feature_cols].fillna(0)

    college_model = Ridge(alpha=1.0).fit(ct_X, ct_y_resid)

    compare = pd.DataFrame({
        "statewide (section 11)": pd.Series(driver_model.coef_, index=driver_feature_cols),
        "Iowa City / Ames only": pd.Series(college_model.coef_, index=driver_feature_cols),
    })
    local_event_rows = ["iowa_state_fair_days", "hawkeyes_home_game_days", "cyclones_home_game_days"]
    print("\nlocal-event coefficients, statewide vs. Iowa City/Ames-only:")
    compare.loc[[c for c in local_event_rows if c in compare.index]].round(3)
else:
    print("too few rows in these zips after filtering to fit a meaningful comparison model")

## 13. 2026 actual vs. predicted, by model (statewide, section-8 models)

In [ ]:
plot_df = pd.concat([dev, test]).copy()
for label, m in models.items():
    pred_log = m.predict(plot_df[feature_cols].fillna(0))
    plot_df[f"pred_percapita__{label}"] = np.expm1(pred_log)

months_order = sorted(plot_df["year_month"].unique())
monthly_actual = plot_df.groupby("year_month")["liters_per_capita"].mean().loc[months_order]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(months_order))
ax.plot(x, monthly_actual.values, marker="o", linewidth=2.5, color="black", label="Actual")
for label in models:
    monthly_pred = plot_df.groupby("year_month")[f"pred_percapita__{label}"].mean().loc[months_order]
    ax.plot(x, monthly_pred.values, marker="o", label=label)

ax.axvline(len(dev_months) - 0.5, color="gray", linestyle="--", linewidth=1)
ax.text(len(dev_months) - 0.5, ax.get_ylim()[1], "dev | test", ha="center", va="bottom", fontsize=9, color="gray")
ax.set_xticks(x)
ax.set_xticklabels(months_order, rotation=45)
ax.set_ylabel("Mean liters per capita across zips")
ax.set_title("2026: actual vs. predicted per-capita liters, by model")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 5), sharex=True, sharey=True)
lims = [plot_df["log_percapita"].min() - 0.2, plot_df["log_percapita"].max() + 0.2]
for ax, label in zip(axes, models):
    pred_log = np.log1p(plot_df[f"pred_percapita__{label}"])
    ax.scatter(plot_df["log_percapita"], pred_log, alpha=0.25, s=12)
    ax.plot(lims, lims, color="red", linewidth=1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(label)
    ax.set_xlabel("actual log_percapita")
axes[0].set_ylabel("predicted log_percapita")
plt.suptitle("2026 (dev+test): predicted vs. actual, zip-month level")
plt.tight_layout()
plt.show()

## 14. How to present this to DEAD

- **Lead with section 12 for anything about *where*** to focus outreach or programming --
  it's a real, checkable local effect in two specific, named places, not an inference from
  a statewide average.
- **Use section 11 only for *when*** -- statewide seasonal/holiday timing for campaign
  scheduling -- and say "spirits" explicitly, not "alcohol," per section 1b/6.
- **Never use the pooled statewide model to name or rank other zip codes as "problem
  areas."** It has neither the local resolution (section 12's whole point) nor the
  demographic/population precision (section 6) to support that kind of claim responsibly.
- Both sections describe correlational, aggregate, area-level patterns -- not individual
  drinking behavior, and not causation.

## 15. Takeaway for DEAD

- **Section 11's coefficients** are the "what's associated with higher per-capita spirits
  purchases, statewide, month to month" answer, now without the non-actionable zip-baseline
  bar drowning everything else out.
- **Section 12** recovers the Iowa City / Ames football/fair effect that section 11's
  pooled model structurally cannot see -- use it for targeting, not the statewide chart.
- **Section 1b's finding matters more than any single coefficient**: if this data is
  spirits-only, every claim here needs that qualifier, and beer/wine-driven occasions
  (Super Bowl Sunday being the clearest example) will always look artificially weak.
- The next highest-value addition, if DEAD wants to keep going, is **ABV/proof data** --
  per-capita liters is a real improvement over raw totals, but it's still liquid volume,
  not ethanol.

In [ ]:
zip_ranking = (
    zip_month.groupby("zip")
    .agg(
        avg_liters_per_capita=("liters_per_capita", "mean"),
        avg_total_liters=("total_liters", "mean"),
        avg_population=("population", "mean"),
        n_months=("year_month", "count"),
    )
    .sort_values("avg_liters_per_capita", ascending=False)
)
zip_ranking.head(20)

In [ ]:
top15 = zip_ranking.head(15).sort_values("avg_liters_per_capita")
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top15.index.astype(str), top15["avg_liters_per_capita"])
ax.set_xlabel("Avg. liters per capita (spirits)")
ax.set_title("Top 15 zips by per-capita spirits volume")
plt.tight_layout()
plt.show()

# Using parquet with the demographic data included:

In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

DATA = "liquor_census.parquet"

zip_month = con.execute(f"""
    SELECT
        LEFT(TRIM(store_zip_code), 5) AS zip_code,

        strftime(ordered_on, '%Y-%m') AS year_month,
        YEAR(ordered_on) AS year,
        MONTH(ordered_on) AS month_num,

        -- TARGET
        SUM(sales_dollars) AS total_sales,

        -- Liquor / store information
        SUM(sales_liters) AS total_liters,
        SUM(TRY_CAST(sales_bottles AS DOUBLE)) AS total_bottles,
        COUNT(DISTINCT store_no) AS n_stores,
        COUNT(DISTINCT category_name) AS n_categories,
        AVG(TRY_CAST(state_bottle_retail AS DOUBLE)) AS avg_bottle_retail,

        -- Demographics
        MAX(total_population) AS total_population,
        MAX(median_household_income) AS median_household_income,
        MAX(per_capita_income) AS per_capita_income,
        MAX(unemployment_rate) AS unemployment_rate,
        MAX(poverty_rate) AS poverty_rate,
        MAX(median_age) AS median_age,
        MAX(pct_age_20_34) AS pct_age_20_34

    FROM '{DATA}'

    WHERE store_zip_code IS NOT NULL
      AND ordered_on IS NOT NULL
      AND sales_dollars IS NOT NULL

    GROUP BY
        zip_code,
        year_month,
        year,
        month_num

    ORDER BY
        year,
        month_num,
        zip_code
""").df()

zip_month.head()

In [ ]:
import pandas as pd

calendar_monthly = pd.read_parquet(
    "calendar_monthly.parquet"
)

In [ ]:
zip_month = zip_month.merge(
    calendar_monthly.drop(columns="date"),
    on=["year", "month_num"],
    how="left"
)

# Look into the top 15 zip codes
- who is here? unemployment rate? population? ethnicity?
- where are these zip codes located?
- any shared characterisitcs?
- look at median income
- and significant dates?

In [ ]:
zip_ranking = (
    zip_month.groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        total_sales=("total_sales", "sum"),
        avg_population=("total_population", "mean"),
        n_months=("year_month", "count"),
    )
    .sort_values("avg_monthly_sales", ascending=False)
)

zip_ranking.head(20)

In [ ]:
top15 = (
    zip_ranking
    .head(15)
    .sort_values("avg_monthly_sales")
)

fig, ax = plt.subplots(figsize=(7, 6))

ax.barh(
    top15.index.astype(str),
    top15["avg_monthly_sales"]
)

ax.set_xlabel("Average Monthly Alcohol Sales ($)")
ax.set_ylabel("ZIP Code")
ax.set_title("Top 15 ZIP Codes by Average Monthly Alcohol Sales")

plt.tight_layout()
plt.show()

## sales per capita

In [ ]:
zip_month["sales_per_capita"] = (
    zip_month["total_sales"] / zip_month["total_population"]
)

zip_ranking = (
    zip_month.groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        total_sales=("total_sales", "sum"),
        avg_population=("total_population", "mean"),
        n_months=("year_month", "count"),
    )
    .sort_values("avg_monthly_sales", ascending=False)
)

zip_ranking.head(20)

| ZIP Code  | City            |
| --------- | --------------- |
| **50314** | Des Moines      |
| **50320** | Des Moines      |
| **50266** | West Des Moines |
| **51501** | Council Bluffs  |
| **52240** | Iowa City       |
| **52807** | Davenport       |
| **50010** | Ames            |
| **52402** | Cedar Rapids    |
| **52241** | Coralville      |
| **50021** | Ankeny          |
| **50613** | Cedar Falls     |
| **52742** | DeWitt          |
| **50401** | Mason City      |
| **52404** | Cedar Rapids    |
| **50311** | Des Moines      |
| **50702** | Waterloo        |
| **52001** | Dubuque         |
| **50322** | Urbandale       |
| **51106** | Sioux City      |
| **52722** | Bettendorf      |


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import requests

# ---------------------------------------------------------
# Top 15 ZIP codes
# ---------------------------------------------------------

top15 = zip_ranking.head(15).reset_index()

top15["zip_code"] = (
    top15["zip_code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

# ---------------------------------------------------------
# Load Iowa ZIP/ZCTA boundaries
# ---------------------------------------------------------

url = "https://raw.githubusercontent.com/OpenDataDE/State-zip-code-GeoJSON/master/ia_iowa_zip_codes_geo.min.json"
iowa_geojson = requests.get(url).json()

# Get all Iowa ZIP codes from GeoJSON
all_zips = [
    feature["properties"]["ZCTA5CE10"]
    for feature in iowa_geojson["features"]
]

# ---------------------------------------------------------
# Base layer: ALL Iowa ZIP boundaries
# ---------------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Choropleth(
        geojson=iowa_geojson,
        locations=all_zips,
        featureidkey="properties.ZCTA5CE10",
        z=[0] * len(all_zips),

        colorscale=[
            [0, "white"],
            [1, "white"]
        ],

        showscale=False,

        marker_line_color="gray",
        marker_line_width=0.6,

        hoverinfo="skip"
    )
)

# ---------------------------------------------------------
# Top 15 layer: highlight by alcohol sales
# ---------------------------------------------------------

fig.add_trace(
    go.Choropleth(
        geojson=iowa_geojson,
        locations=top15["zip_code"],
        featureidkey="properties.ZCTA5CE10",
        z=top15["avg_monthly_sales"],

        colorscale="Reds",

        marker_line_color="black",
        marker_line_width=1.2,

        colorbar_title="Avg. Monthly<br>Sales ($)",

        customdata=top15[
            [
                "avg_monthly_sales",
                "total_sales",
                "avg_population"
            ]
        ],

        hovertemplate=(
            "<b>ZIP %{location}</b><br>"
            "Avg. Monthly Sales: $%{customdata[0]:,.0f}<br>"
            "Total Sales: $%{customdata[1]:,.0f}<br>"
            "Avg. Population: %{customdata[2]:,.0f}"
            "<extra></extra>"
        )
    )
)

# ---------------------------------------------------------
# Display entire state of Iowa
# ---------------------------------------------------------

fig.update_geos(
    fitbounds="locations",
    visible=False,
    projection_type="mercator"
)

fig.update_layout(
    title="Top 15 Iowa ZIP Codes by Average Monthly Alcohol Sales",
    margin=dict(l=0, r=0, t=50, b=0),
    height=650
)

fig.show()

Who lives in these ZIPs? Are they large-population areas? Higher/lower income? Younger? Higher poverty/unemployment?

In [ ]:
top15_zips = zip_ranking.head(15).index.astype(str).tolist()

top15_demo = (
    zip_month[
        zip_month["zip_code"].astype(str).isin(top15_zips)
    ]
    .groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        population=("total_population", "mean"),
        median_income=("median_household_income", "mean"),
        per_capita_income=("per_capita_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
        n_stores=("n_stores", "mean")
    )
    .sort_values("avg_monthly_sales", ascending=False)
)

top15_demo

top 15 to allzip code comparison

In [ ]:
zip_month["top15"] = (
    zip_month["zip_code"].astype(str).isin(top15_zips)
)

comparison = (
    zip_month
    .groupby("top15")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_population=("total_population", "mean"),
        avg_median_income=("median_household_income", "mean"),
        avg_unemployment=("unemployment_rate", "mean"),
        avg_poverty=("poverty_rate", "mean"),
        avg_median_age=("median_age", "mean"),
        avg_pct_age_20_34=("pct_age_20_34", "mean"),
        avg_n_stores=("n_stores", "mean")
    )
)

comparison.index = ["Other ZIPs", "Top 15 ZIPs"]

comparison

In [ ]:
top15_yearly = (
    zip_month[
        (zip_month["zip_code"].astype(str).isin(top15_zips)) &
        (zip_month["year"].isin([2022, 2023, 2024, 2025]))
    ]
    .groupby(["zip_code", "year"])
    .agg(
        total_sales=("total_sales", "sum"),
        avg_monthly_sales=("total_sales", "mean")
    )
    .reset_index()
)

# top15_yearly

In [ ]:
import matplotlib.pyplot as plt

for zip_code in top15_zips:
    temp = top15_yearly[
        top15_yearly["zip_code"].astype(str) == str(zip_code)
    ]

    plt.plot(
        temp["year"],
        temp["avg_monthly_sales"],
        marker="o",
        label=zip_code
    )

plt.xlabel("Year")
plt.ylabel("Average Monthly Alcohol Sales ($)")
plt.title("Alcohol Sales in Top 15 ZIP Codes, 2022–2025")
plt.xticks([2022, 2023, 2024, 2025])
plt.legend(title="ZIP", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
events = [
    "new_years_day",
    "new_years_eve",
    "super_bowl_sunday",
    "st_patricks_day",
    "cinco_de_mayo",
    "memorial_day",
    "july_4th",
    "labor_day",
    "halloween",
    "thanksgiving",
    "christmas",
    "iowa_state_fair",
    "hawkeyes_home_game",
    "cyclones_home_game"
]

top15_data = zip_month[
    zip_month["zip_code"].astype(str).isin(top15_zips)
]

event_results = []

for event in events:
    if event in top15_data.columns:

        event_sales = top15_data.loc[
            top15_data[event] == 1,
            "total_sales"
        ].mean()

        non_event_sales = top15_data.loc[
            top15_data[event] == 0,
            "total_sales"
        ].mean()

        event_results.append({
            "event": event,
            "event_avg_sales": event_sales,
            "non_event_avg_sales": non_event_sales,
            "difference": event_sales - non_event_sales
        })

event_comparison = pd.DataFrame(event_results)

event_comparison.sort_values(
    "difference",
    ascending=False
)

In [ ]:
top_zips = zip_ranking.head(15).index.astype(str)

zip_month["group"] = np.where(
    zip_month["zip_code"].astype(str).isin(top_zips),
    "Top 15",
    "Other Iowa ZIPs"
)

demo_comparison = (
    zip_month
    .groupby("group")
    .agg(
        population=("total_population", "mean"),
        median_income=("median_household_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
        avg_number_stores=("n_stores", "mean"),
        avg_monthly_sales=("total_sales", "mean")
    )
    .round(2)
)

demo_comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    zip_month["total_population"],
    zip_month["total_sales"],
    alpha=0.4
)

plt.xlabel("Population")
plt.ylabel("Monthly Alcohol Sales ($)")
plt.title("Population vs. Monthly Alcohol Sales")

plt.tight_layout()
plt.show()

high sales relative to population

In [ ]:
zip_month["sales_per_capita"] = (
    zip_month["total_sales"] /
    zip_month["total_population"]
)

In [ ]:
per_capita_ranking = (
    zip_month
    .groupby("zip_code")
    .agg(
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_monthly_sales=("total_sales", "mean"),
        population=("total_population", "mean"),
        avg_stores=("n_stores", "mean")
    )
    .sort_values("avg_sales_per_capita", ascending=False)
)

per_capita_ranking.head(20)

In [ ]:
top15_zips = zip_ranking.head(15).index.astype(str)

top15_monthly = (
    zip_month[
        zip_month["zip_code"].astype(str).isin(top15_zips)
    ]
    .groupby("month_num")
    .agg(
        avg_sales=("total_sales", "mean")
    )
    .reset_index()
)

top15_monthly

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    top15_monthly["month_num"],
    top15_monthly["avg_sales"],
    marker="o"
)

plt.xticks(
    range(1, 13),
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)

plt.xlabel("Month")
plt.ylabel("Average Monthly Alcohol Sales ($)")
plt.title("Average Alcohol Sales in Top 15 Iowa ZIP Codes by Month")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Top 15 ZIP codes
top15_zips = zip_ranking.head(15).index.astype(str)

# Keep top 15 ZIPs
top15_data = zip_month[
    zip_month["zip_code"].astype(str).isin(top15_zips)
].copy()

# Average sales across the top 15 ZIPs for each month/year
monthly_sales = (
    top15_data
    .groupby(["year", "month_num"])
    .agg(avg_sales=("total_sales", "mean"))
    .reset_index()
)

month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# Make one plot for each year
for year in [2022, 2023, 2024, 2025, 2026]:

    data = monthly_sales[
        monthly_sales["year"] == year
    ].sort_values("month_num")

    plt.figure(figsize=(9, 4))

    plt.plot(
        data["month_num"],
        data["avg_sales"],
        marker="o"
    )

    plt.xticks(
        range(1, 13),
        month_names
    )

    plt.xlabel("Month")
    plt.ylabel("Average Monthly Alcohol Sales ($)")

    if year == 2026:
        plt.title(
            "Average Alcohol Sales in Top 15 ZIP Codes - 2026 (YTD)"
        )
    else:
        plt.title(
            f"Average Alcohol Sales in Top 15 ZIP Codes - {year}"
        )

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ============================================================
# PREPARE DATA
# ============================================================

df = zip_month.copy()

# Only use complete years for model training/testing
df = df[df["year"].isin([2022, 2023, 2024, 2025])].copy()

# Month should be categorical, not treated as a numeric 1-12 scale
month_dummies = pd.get_dummies(
    df["month_num"],
    prefix="month",
    drop_first=True,
    dtype=int
)

df = pd.concat([df, month_dummies], axis=1)

month_features = list(month_dummies.columns)

target = "total_sales"


# ============================================================
# CANDIDATE MODELS
# ============================================================

# Model 1:
# How much can market size alone explain?
market_model = [
    "total_population",
    "n_stores"
]

# Model 2:
# Do demographics add information beyond market size?
demographic_model = market_model + [
    "median_household_income",
    "unemployment_rate",
    "poverty_rate",
    "median_age",
    "pct_age_20_34"
]

# Model 3:
# Does WHEN (seasonality) improve the model?
season_model = demographic_model + month_features

# Model 4:
# Do special Iowa events add additional information?
event_model = season_model + [
    "iowa_state_fair",
    "hawkeyes_home_game",
    "cyclones_home_game"
]

candidate_models = {
    "Market Size": market_model,
    "Demographics": demographic_model,
    "Seasonality": season_model,
    "Seasonality + Events": event_model
}


# ============================================================
# MANUAL TIME-BASED VALIDATION
#
# Fold 1: train 2022 -> validate 2023
# Fold 2: train 2022-2023 -> validate 2024
# ============================================================

folds = [
    ([2022], 2023),
    ([2022, 2023], 2024)
]

validation_results = []

for model_name, features in candidate_models.items():

    for train_years, val_year in folds:

        train_fold = df[
            df["year"].isin(train_years)
        ].dropna(subset=features + [target])

        val_fold = df[
            df["year"] == val_year
        ].dropna(subset=features + [target])

        X_train = train_fold[features]
        y_train = train_fold[target]

        X_val = val_fold[features]
        y_val = val_fold[target]

        model = LinearRegression()
        model.fit(X_train, y_train)

        pred = model.predict(X_val)

        validation_results.append({
            "model": model_name,
            "validation_year": val_year,
            "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
            "MAE": mean_absolute_error(y_val, pred),
            "R2": r2_score(y_val, pred)
        })


# ============================================================
# COMPARE CANDIDATE MODELS
# ============================================================

validation_results = pd.DataFrame(validation_results)

model_comparison = (
    validation_results
    .groupby("model")
    .agg(
        avg_RMSE=("RMSE", "mean"),
        avg_MAE=("MAE", "mean"),
        avg_R2=("R2", "mean")
    )
    .sort_values("avg_RMSE")
)

print("\n================ MODEL SELECTION ================\n")
print(model_comparison.round(3))


# ============================================================
# SELECT BEST MODEL BASED ON LOWEST VALIDATION RMSE
# ============================================================

best_model_name = model_comparison.index[0]
best_features = candidate_models[best_model_name]

print("\nSelected Model:", best_model_name)

print("\nSelected Features:")
for feature in best_features:
    print(" -", feature)


# ============================================================
# FINAL MODEL
#
# Train: 2022-2024
# Test:  2025
# ============================================================

train = df[
    df["year"].isin([2022, 2023, 2024])
].dropna(subset=best_features + [target]).copy()

test = df[
    df["year"] == 2025
].dropna(subset=best_features + [target]).copy()

X_train = train[best_features]
y_train = train[target]

X_test = test[best_features]
y_test = test[target]

final_model = LinearRegression()

# TRAIN MODEL
final_model.fit(X_train, y_train)

# Predict unseen 2025 data
test["predicted_sales"] = final_model.predict(X_test)


# ============================================================
# FINAL TEST PERFORMANCE
# ============================================================

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test["predicted_sales"]
    )
)

test_mae = mean_absolute_error(
    y_test,
    test["predicted_sales"]
)

test_r2 = r2_score(
    y_test,
    test["predicted_sales"]
)

print("\n================ 2025 TEST RESULTS ================\n")

print(f"RMSE: ${test_rmse:,.2f}")
print(f"MAE:  ${test_mae:,.2f}")
print(f"R²:   {test_r2:.3f}")


# ============================================================
# COEFFICIENTS
# ============================================================

coefficients = pd.DataFrame({
    "feature": best_features,
    "coefficient": final_model.coef_
})

print("\n================ COEFFICIENTS ================\n")

print(
    coefficients
    .sort_values("coefficient", ascending=False)
    .to_string(index=False)
)


# ============================================================
# RESIDUALS
#
# Positive residual:
# actual sales > model expected
# ============================================================

test["residual"] = (
    test["total_sales"] -
    test["predicted_sales"]
)


# ============================================================
# WHERE?
# Which ZIPs repeatedly had higher sales than expected in 2025?
# ============================================================

where_results = (
    test
    .groupby("zip_code")
    .agg(
        avg_actual_sales=("total_sales", "mean"),
        avg_expected_sales=("predicted_sales", "mean"),
        avg_residual=("residual", "mean"),
        months_above_expected=(
            "residual",
            lambda x: (x > 0).sum()
        ),
        total_population=("total_population", "mean")
    )
    .reset_index()
)

# Per-capita purchasing gives additional context
where_results["sales_per_capita"] = (
    where_results["avg_actual_sales"] /
    where_results["total_population"]
)

where_results = where_results.sort_values(
    "avg_residual",
    ascending=False
)

print("\n================ WHERE? ================\n")
print(
    where_results[
        [
            "zip_code",
            "avg_actual_sales",
            "sales_per_capita",
            "avg_residual",
            "months_above_expected"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)


# ============================================================
# WHEN?
# Which months had higher sales than expected?
# ============================================================

when_results = (
    test
    .groupby("month_num")
    .agg(
        avg_actual_sales=("total_sales", "mean"),
        avg_expected_sales=("predicted_sales", "mean"),
        avg_residual=("residual", "mean")
    )
    .reset_index()
)

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

when_results["month"] = (
    when_results["month_num"].map(month_names)
)

when_results["pct_above_expected"] = (
    when_results["avg_residual"] /
    when_results["avg_expected_sales"]
) * 100

when_results = when_results.sort_values(
    "avg_residual",
    ascending=False
)

print("\n================ WHEN? ================\n")

print(
    when_results[
        [
            "month",
            "avg_actual_sales",
            "avg_expected_sales",
            "avg_residual",
            "pct_above_expected"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


# ============================================================
# WHERE + WHEN?
#
# Largest ZIP-month deviations in 2025
# ============================================================

priority_periods = (
    test[
        [
            "zip_code",
            "year",
            "month_num",
            "total_sales",
            "predicted_sales",
            "residual",
            "total_population"
        ]
    ]
    .copy()
)

priority_periods["month"] = (
    priority_periods["month_num"].map(month_names)
)

priority_periods["sales_per_capita"] = (
    priority_periods["total_sales"] /
    priority_periods["total_population"]
)

priority_periods["pct_above_expected"] = (
    priority_periods["residual"] /
    priority_periods["predicted_sales"]
) * 100

priority_periods = priority_periods.sort_values(
    "residual",
    ascending=False
)

print("\n================ WHERE + WHEN? ================\n")

print(
    priority_periods[
        [
            "zip_code",
            "month",
            "total_sales",
            "sales_per_capita",
            "residual",
            "pct_above_expected"
        ]
    ]
    .head(25)
    .round(2)
    .to_string(index=False)
)

which ZIP codes show elevated alcohol purchasing, and during which months does that pattern occur?

It uses three different signals for each ZIP:

Volume: Is average monthly alcohol sales high?
Per capita: Is purchasing high relative to the ZIP's population?
Higher than expected: Does the ZIP have higher sales than your regression model predicts?

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# CAMPAIGN PRIORITY ANALYSIS
# Uses complete years: 2022-2025
# ============================================================

campaign = df[
    df["year"].isin([2022, 2023, 2024, 2025])
].copy()

campaign = campaign.dropna(
    subset=best_features + ["total_sales", "total_population"]
)

# ------------------------------------------------------------
# 1. MODEL-EXPECTED SALES
# ------------------------------------------------------------

campaign["predicted_sales"] = final_model.predict(
    campaign[best_features]
)

campaign["residual"] = (
    campaign["total_sales"] -
    campaign["predicted_sales"]
)

# Positive = more purchasing than model expected
campaign["above_expected"] = (
    campaign["residual"] > 0
).astype(int)


# ------------------------------------------------------------
# 2. SALES PER CAPITA
# ------------------------------------------------------------

campaign["sales_per_capita"] = (
    campaign["total_sales"] /
    campaign["total_population"]
)


# ============================================================
# WHERE?
# Summarize each ZIP across 2022-2025
# ============================================================

zip_priority = (
    campaign
    .groupby("zip_code")
    .agg(
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        months_above_expected=("above_expected", "sum"),
        months_observed=("above_expected", "count"),
        population=("total_population", "mean")
    )
    .reset_index()
)

# Percentage of months where sales exceeded model expectations
zip_priority["pct_months_above_expected"] = (
    zip_priority["months_above_expected"] /
    zip_priority["months_observed"] * 100
)


# ------------------------------------------------------------
# 3. DEFINE "HIGH" USING TOP 25%
# ------------------------------------------------------------

sales_cutoff = zip_priority["avg_monthly_sales"].quantile(.75)

percap_cutoff = zip_priority["avg_sales_per_capita"].quantile(.75)

residual_cutoff = zip_priority["avg_residual"].quantile(.75)


zip_priority["high_volume"] = (
    zip_priority["avg_monthly_sales"] >= sales_cutoff
).astype(int)

zip_priority["high_per_capita"] = (
    zip_priority["avg_sales_per_capita"] >= percap_cutoff
).astype(int)

zip_priority["high_residual"] = (
    zip_priority["avg_residual"] >= residual_cutoff
).astype(int)


# ------------------------------------------------------------
# 4. NUMBER OF ELEVATED-PURCHASING SIGNALS
#
# 3 = high volume + high per capita + high residual
# 2 = two indicators
# 1 = one indicator
# 0 = none
# ------------------------------------------------------------

zip_priority["priority_signals"] = (
    zip_priority["high_volume"] +
    zip_priority["high_per_capita"] +
    zip_priority["high_residual"]
)

zip_priority = zip_priority.sort_values(
    [
        "priority_signals",
        "pct_months_above_expected",
        "avg_monthly_sales"
    ],
    ascending=False
)


print("\n================================================")
print("WHERE: CAMPAIGN-PRIORITY ZIP CODES")
print("================================================\n")

print(
    zip_priority[
        [
            "zip_code",
            "avg_monthly_sales",
            "avg_sales_per_capita",
            "avg_residual",
            "pct_months_above_expected",
            "priority_signals"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)


# ============================================================
# 5. KEEP ZIPS WITH MULTIPLE SIGNALS
# ============================================================

priority_zips = zip_priority.loc[
    zip_priority["priority_signals"] >= 2,
    "zip_code"
].astype(str)

priority_data = campaign[
    campaign["zip_code"].astype(str).isin(priority_zips)
].copy()


# ============================================================
# WHEN?
# Find recurring monthly patterns in priority ZIPs
# ============================================================

month_priority = (
    priority_data
    .groupby("month_num")
    .agg(
        avg_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        pct_observations_above_expected=(
            "above_expected",
            "mean"
        )
    )
    .reset_index()
)

month_priority["pct_observations_above_expected"] *= 100

month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

month_priority["month"] = (
    month_priority["month_num"].map(month_names)
)

month_priority = month_priority.sort_values(
    "avg_sales_per_capita",
    ascending=False
)


print("\n================================================")
print("WHEN: CAMPAIGN-PRIORITY MONTHS")
print("================================================\n")

print(
    month_priority[
        [
            "month",
            "avg_sales",
            "avg_sales_per_capita",
            "avg_residual",
            "pct_observations_above_expected"
        ]
    ]
    .round(2)
    .to_string(index=False)
)


# ============================================================
# 6. WHERE + WHEN TOGETHER
# Average each ZIP + calendar month across years
# ============================================================

zip_month_priority = (
    priority_data
    .groupby(["zip_code", "month_num"])
    .agg(
        avg_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        years_above_expected=("above_expected", "sum"),
        years_observed=("above_expected", "count")
    )
    .reset_index()
)

zip_month_priority["month"] = (
    zip_month_priority["month_num"].map(month_names)
)

zip_month_priority["pct_years_above_expected"] = (
    zip_month_priority["years_above_expected"] /
    zip_month_priority["years_observed"] * 100
)


# ------------------------------------------------------------
# 7. IDENTIFY RECURRING ZIP-MONTH PATTERNS
#
# Require:
# - positive average residual
# - above expected in >= 75% of observed years
# ------------------------------------------------------------

campaign_targets = (
    zip_month_priority[
        (zip_month_priority["avg_residual"] > 0) &
        (zip_month_priority["pct_years_above_expected"] >= 75)
    ]
    .sort_values(
        [
            "pct_years_above_expected",
            "avg_sales_per_capita",
            "avg_sales"
        ],
        ascending=False
    )
)


print("\n================================================")
print("WHERE + WHEN: RECURRING ELEVATED ZIP-MONTHS")
print("================================================\n")

print(
    campaign_targets[
        [
            "zip_code",
            "month",
            "avg_sales",
            "avg_sales_per_capita",
            "avg_residual",
            "pct_years_above_expected"
        ]
    ]
    .head(30)
    .round(2)
    .to_string(index=False)
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# 1. TABLE: TOP CAMPAIGN-PRIORITY ZIP CODES
# ============================================================

priority_table = (
    zip_priority[
        [
            "zip_code",
            "avg_monthly_sales",
            "avg_sales_per_capita",
            "avg_residual",
            "pct_months_above_expected",
            "priority_signals"
        ]
    ]
    .sort_values(
        ["priority_signals", "avg_sales_per_capita"],
        ascending=False
    )
    .head(15)
    .copy()
)

print("\nTOP CAMPAIGN-PRIORITY ZIP CODES\n")

display(
    priority_table.style.format({
        "avg_monthly_sales": "${:,.0f}",
        "avg_sales_per_capita": "${:,.2f}",
        "avg_residual": "${:,.0f}",
        "pct_months_above_expected": "{:.1f}%"
    })
)


# ============================================================
# 2. GRAPH: PER-CAPITA SALES IN PRIORITY ZIP CODES
# ============================================================

plot_data = priority_table.sort_values(
    "avg_sales_per_capita"
)

plt.figure(figsize=(10, 7))

plt.barh(
    plot_data["zip_code"].astype(str),
    plot_data["avg_sales_per_capita"]
)

plt.xlabel("Average Monthly Sales Per Capita ($)")
plt.ylabel("ZIP Code")
plt.title("Alcohol Purchasing Intensity in Campaign-Priority ZIP Codes")

plt.tight_layout()
plt.show()


# ============================================================
# 3. TABLE: MONTHLY PATTERNS IN PRIORITY ZIP CODES
# ============================================================

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

monthly_table = month_priority.copy()

monthly_table["month"] = pd.Categorical(
    monthly_table["month"],
    categories=month_order,
    ordered=True
)

monthly_table = monthly_table.sort_values("month")

print("\nMONTHLY PATTERNS IN PRIORITY ZIP CODES\n")

display(
    monthly_table[
        [
            "month",
            "avg_sales",
            "avg_sales_per_capita",
            "avg_residual",
            "pct_observations_above_expected"
        ]
    ].style.format({
        "avg_sales": "${:,.0f}",
        "avg_sales_per_capita": "${:,.2f}",
        "avg_residual": "${:,.0f}",
        "pct_observations_above_expected": "{:.1f}%"
    })
)


# ============================================================
# 4. GRAPH: MONTHLY PURCHASING PATTERN
# ============================================================

plt.figure(figsize=(11, 5))

plt.plot(
    monthly_table["month"],
    monthly_table["avg_sales_per_capita"],
    marker="o"
)

plt.xlabel("Month")
plt.ylabel("Average Sales Per Capita ($)")
plt.title("When Is Alcohol Purchasing Highest in Priority ZIP Codes?")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ============================================================
# 5. CREATE ZIP × MONTH PATTERN
#
# Instead of comparing raw dollars, compare each ZIP-month
# with that ZIP's normal monthly sales-per-capita level.
# ============================================================

heatmap_data = priority_data.copy()

# Normal sales-per-capita level for each ZIP
heatmap_data["zip_normal"] = (
    heatmap_data
    .groupby("zip_code")["sales_per_capita"]
    .transform("mean")
)

# % above/below what is normal for that ZIP
heatmap_data["pct_from_zip_normal"] = (
    (
        heatmap_data["sales_per_capita"]
        / heatmap_data["zip_normal"]
    ) - 1
) * 100


# Average the same calendar month across 2022-2025
zip_month_pattern = (
    heatmap_data
    .groupby(["zip_code", "month_num"])
    .agg(
        avg_pct_from_normal=("pct_from_zip_normal", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_total_sales=("total_sales", "mean")
    )
    .reset_index()
)


# ============================================================
# 6. HEATMAP: WHERE + WHEN
# ============================================================

# Keep strongest ZIPs for readable presentation
top_zips = (
    zip_priority
    .sort_values(
        ["priority_signals", "avg_monthly_sales"],
        ascending=False
    )
    .head(15)["zip_code"]
)

heatmap_plot = (
    zip_month_pattern[
        zip_month_pattern["zip_code"].isin(top_zips)
    ]
    .pivot(
        index="zip_code",
        columns="month_num",
        values="avg_pct_from_normal"
    )
)

# Make sure months are Jan -> Dec
heatmap_plot = heatmap_plot.reindex(
    columns=range(1, 13)
)

heatmap_plot.columns = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

plt.figure(figsize=(12, 8))

sns.heatmap(
    heatmap_plot,
    annot=True,
    fmt=".0f",
    center=0,
    cmap="RdBu_r",
    cbar_kws={
        "label": "% Above/Below ZIP's Normal Purchasing"
    }
)

plt.xlabel("Month")
plt.ylabel("ZIP Code")
plt.title(
    "Where and When Is Alcohol Purchasing Elevated?\n"
    "Average Pattern, 2022–2025"
)

plt.tight_layout()
plt.show()


# ============================================================
# 7. FINAL WHERE + WHEN TABLE
# ============================================================

pattern_table = (
    zip_month_pattern[
        zip_month_pattern["zip_code"].isin(top_zips)
    ]
    .copy()
)

pattern_table["month"] = pattern_table["month_num"].map({
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
})

# Focus on months above the ZIP's normal purchasing level
pattern_table = (
    pattern_table[
        pattern_table["avg_pct_from_normal"] > 0
    ]
    .sort_values(
        "avg_pct_from_normal",
        ascending=False
    )
)

print("\nSTRONGEST RECURRING WHERE + WHEN PATTERNS\n")

display(
    pattern_table[
        [
            "zip_code",
            "month",
            "avg_total_sales",
            "avg_sales_per_capita",
            "avg_pct_from_normal"
        ]
    ]
    .head(30)
    .style.format({
        "avg_total_sales": "${:,.0f}",
        "avg_sales_per_capita": "${:,.2f}",
        "avg_pct_from_normal": "+{:.1f}%"
    })
)

In [ ]:
# ============================================================
# DEAD: WHERE ARE ELEVATED PURCHASING AREAS,
# AND WHAT ARE THEIR DEMOGRAPHIC CHARACTERISTICS?
# ============================================================

zip_context = (
    campaign
    .groupby("zip_code")
    .agg(
        # Purchasing
        avg_monthly_sales=("total_sales", "mean"),
        avg_sales_per_capita=("sales_per_capita", "mean"),
        avg_residual=("residual", "mean"),
        pct_months_above_expected=(
            "above_expected",
            lambda x: x.mean() * 100
        ),

        # Population / access
        population=("total_population", "mean"),
        avg_stores=("n_stores", "mean"),

        # Socioeconomic characteristics
        median_income=("median_household_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),

        # Age
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean")
    )
    .reset_index()
)


# ============================================================
# PURCHASING SIGNALS
# ============================================================

volume_cut = zip_context["avg_monthly_sales"].quantile(.75)

percap_cut = zip_context["avg_sales_per_capita"].quantile(.75)

residual_cut = zip_context["avg_residual"].quantile(.75)


zip_context["high_volume"] = (
    zip_context["avg_monthly_sales"] >= volume_cut
).astype(int)

zip_context["high_per_capita"] = (
    zip_context["avg_sales_per_capita"] >= percap_cut
).astype(int)

zip_context["higher_than_expected"] = (
    zip_context["avg_residual"] >= residual_cut
).astype(int)


zip_context["purchasing_signals"] = (
    zip_context["high_volume"]
    + zip_context["high_per_capita"]
    + zip_context["higher_than_expected"]
)


# ============================================================
# KEEP AREAS WITH AT LEAST 2 PURCHASING SIGNALS
# ============================================================

priority_context = (
    zip_context[
        zip_context["purchasing_signals"] >= 2
    ]
    .sort_values(
        [
            "purchasing_signals",
            "avg_sales_per_capita"
        ],
        ascending=False
    )
)


# ============================================================
# DISPLAY TABLE
# ============================================================

display(
    priority_context[
        [
            "zip_code",
            "avg_monthly_sales",
            "avg_sales_per_capita",
            "purchasing_signals",
            "population",
            "median_income",
            "unemployment_rate",
            "poverty_rate",
            "median_age",
            "pct_age_20_34",
            "avg_stores"
        ]
    ]
    .head(20)
    .style.format({
        "avg_monthly_sales": "${:,.0f}",
        "avg_sales_per_capita": "${:,.2f}",
        "population": "{:,.0f}",
        "median_income": "${:,.0f}",
        "unemployment_rate": "{:.1f}%",
        "poverty_rate": "{:.1f}%",
        "median_age": "{:.1f}",
        "pct_age_20_34": "{:.1f}%",
        "avg_stores": "{:.1f}"
    })
)

In [ ]:
# ============================================================
# PRIORITY AREAS VS OTHER IOWA ZIP CODES
# ============================================================

zip_context["group"] = np.where(
    zip_context["purchasing_signals"] >= 2,
    "Campaign-priority ZIPs",
    "Other Iowa ZIPs"
)

group_comparison = (
    zip_context
    .groupby("group")
    .agg(
        avg_monthly_sales=("avg_monthly_sales", "mean"),
        sales_per_capita=("avg_sales_per_capita", "mean"),

        population=("population", "mean"),
        median_income=("median_income", "mean"),
        unemployment_rate=("unemployment_rate", "mean"),
        poverty_rate=("poverty_rate", "mean"),
        median_age=("median_age", "mean"),
        pct_age_20_34=("pct_age_20_34", "mean"),
        avg_stores=("avg_stores", "mean")
    )
    .round(2)
)

display(group_comparison)